# Lab 5 Task 1: Deterministic OFDM Transmitter

Complete the OFDM accounting, build the deterministic frame, verify the QPSK and cyclic-prefix round trips, and inspect the supplied frame and resource-grid plots.


In [ ]:
from pathlib import Path
import json
import os
import sys
import numpy as np
from IPython.display import Image, display
import IPython.core.pylabtools as pylabtools

# Matplotlib 3.6 expects this compatibility table, which newer IPython omits.
if not hasattr(pylabtools, "backend2gui"):
    pylabtools.backend2gui = {}

def locate_lab_root():
    for base in (Path.cwd(), *Path.cwd().parents):
        if (base / "support" / "lab5_dsp.py").is_file():
            return base
    raise FileNotFoundError("Open this notebook from its Lab 5 folder.")

LAB_ROOT = locate_lab_root()
sys.path.insert(0, str(LAB_ROOT / "support"))
WORK_DIR = LAB_ROOT / "work"
FIGURE_DIR = WORK_DIR / "figures"
WAVEFORM_DIR = WORK_DIR / "waveforms"
CAPTURE_DIR = Path(os.environ.get("ECE7377_LAB5_CAPTURE_DIR", WORK_DIR / "captures"))
for path in (FIGURE_DIR, WAVEFORM_DIR, CAPTURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

import lab5_dsp as lab
import lab5_tools

print("Lab root:", LAB_ROOT)
print("Capture directory:", CAPTURE_DIR)


## OFDM timing and carrier calculations


In [ ]:
sample_rate_sps = 1_000_000
nfft = 64
ncp = 16
active_carriers = 52
pilot_carriers = 4
payload_symbols = 20

# To do: derive these quantities from the values above.
subcarrier_spacing_hz = ...
useful_symbol_seconds = ...
cp_seconds = ...
total_symbol_seconds = ...
data_carriers = ...
bits_per_payload_symbol = ...
bits_per_frame = ...

signed_examples = (-26, -21, -1, 1, 21, 26)
array_indices = ...

print("subcarrier spacing (Hz):", subcarrier_spacing_hz)
print("useful, CP, total durations (us):", 1e6*useful_symbol_seconds, 1e6*cp_seconds, 1e6*total_symbol_seconds)
print("data carriers, bits/symbol, bits/frame:", data_carriers, bits_per_payload_symbol, bits_per_frame)
print("signed carrier to array index:", array_indices)


In [ ]:
assert subcarrier_spacing_hz == 15_625
assert np.isclose(useful_symbol_seconds, 64e-6)
assert np.isclose(cp_seconds, 16e-6)
assert np.isclose(total_symbol_seconds, 80e-6)
assert data_carriers == 48
assert bits_per_payload_symbol == 96
assert bits_per_frame == 1920
assert array_indices == {-26: 38, -21: 43, -1: 63, 1: 1, 21: 21, 26: 26}
print("Task 1 calculations passed.")


## Construct and verify the frame


In [ ]:
reference = lab.build_reference_frame(sequence_number=0)
ideal = lab.ideal_round_trip(reference)
payload_cp = lab.ofdm_modulate(reference.payload_grid, reference.ncp)
recovered_grid = lab.ofdm_demodulate(payload_cp, reference.ncp)

print("active frame samples:", reference.active_frame_length)
print("logical period samples:", reference.logical_period.size)
print("TX scale:", reference.tx_scale)
print("PAPR (dB):", reference.papr_db)
print("ideal results:", ideal)
print("maximum CP-copy error:", np.max(np.abs(payload_cp[:, :ncp] - payload_cp[:, -ncp:])))


In [ ]:
assert reference.active_frame_length == 2080
assert reference.logical_period.size == 8192
assert np.max(np.abs(reference.tx_period)) <= 0.1000001
assert np.allclose(recovered_grid, reference.payload_grid, atol=4e-7)
assert ideal["bit_errors"] == 0
print("Deterministic frame checks passed.")


## Supplied plots


In [ ]:
simulation = lab5_tools.generate_simulation(FIGURE_DIR)
for name in ('task1_frame_and_resource_grid', 'task1_ofdm_symbol'):
    path = FIGURE_DIR / f"{name}.png"
    display(Image(filename=path))


## Report work

1. Show the timing, carrier, and bit-count calculations.
2. Explain why signed negative subcarriers map to the upper NumPy indices.
3. Explain why the CP contains the final samples of the useful symbol.
4. Report scale, PAPR, ideal BER, EVM, grid error, and null-carrier error.
5. Explain why clipping would destroy orthogonality even if the FFT implementation is correct.
